# Ollama GPU Fix
Run cells in order. Cell 1 is the key fix — only needs to run once.
Cells 2 onwards need to run every time the notebook restarts.

## Cell 1: Diagnose what CUDA libraries exist on the system (run once)

In [1]:
import subprocess
import os

# Find CUDA libraries on the system
print('=== CUDA lib locations ===')
result = subprocess.run(
    ['find', '/usr', '/opt', '/lib', '-name', 'libcuda.so*', '-o', '-name', 'libcudart.so*'],
    capture_output=True, text=True
)
print(result.stdout[:3000])

print('=== nvidia-smi location ===')
result = subprocess.run(['which', 'nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

print('=== Current LD_LIBRARY_PATH ===')
print(os.environ.get('LD_LIBRARY_PATH', 'not set'))

print('=== CUDA_HOME ===')
print(os.environ.get('CUDA_HOME', 'not set'))
print(os.environ.get('CUDA_PATH', 'not set'))

# Check what Ollama extracted
HOME = os.environ['HOME']
print('=== Ollama bin directory contents ===')
result = subprocess.run(['ls', '-la', f'{HOME}/bin'], capture_output=True, text=True)
print(result.stdout)

print('=== Looking for cuda libs in ~/bin ===')
result = subprocess.run(
    ['find', f'{HOME}/bin', '-name', '*.so*'],
    capture_output=True, text=True
)
print(result.stdout[:3000] if result.stdout else 'None found')

=== CUDA lib locations ===
/usr/local/cuda-12.8/targets/x86_64-linux/lib/stubs/libcuda.so
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12.8.90
/usr/lib/x86_64-linux-gnu/libcuda.so
/usr/lib/x86_64-linux-gnu/libcuda.so.570.211.01
/usr/lib/x86_64-linux-gnu/libcuda.so.1
/opt/conda/lib/python3.13/site-packages/nvidia/cuda_runtime/lib/libcudart.so.12

=== nvidia-smi location ===
/usr/bin/nvidia-smi

=== Current LD_LIBRARY_PATH ===
/usr/local/cuda/lib64
=== CUDA_HOME ===
not set
not set
=== Ollama bin directory contents ===
total 41824
drwxrwxr-x 2 nobody nogroup     4096 Apr  3 06:05 .
drwxr-x--- 2 nobody nogroup     4096 Apr  3 16:36 ..
-rwxr-xr-x 1 nobody nogroup 42827368 Mar 30 02:31 ollama

=== Looking for cuda libs in ~/bin ===
None found


## Cell 2: Re-extract Ollama with ALL libraries (run once after Cell 1 confirms paths)

In [2]:
import subprocess
import os

HOME = os.environ['HOME']
zst_path = f'{HOME}/ollama.tar.zst'
OLLAMA_DIR = f'{HOME}/ollama_full'

# Extract EVERYTHING into a dedicated directory
os.makedirs(OLLAMA_DIR, exist_ok=True)
result = subprocess.run(
    ['tar', '--use-compress-program=zstd', '-xf', zst_path, '-C', OLLAMA_DIR],
    capture_output=True, text=True
)
print('Extract result:', result.returncode)

# Show what was extracted
result = subprocess.run(['find', OLLAMA_DIR, '-type', 'f'],
                        capture_output=True, text=True)
print('Extracted files:')
print(result.stdout[:3000])

Extract result: 0
Extracted files:
/home/kbarbee2/ollama_full/bin/ollama
/home/kbarbee2/ollama_full/lib/ollama/cuda_v12/libcublas.so.12.8.4.1
/home/kbarbee2/ollama_full/lib/ollama/cuda_v12/libcublasLt.so.12.8.4.1
/home/kbarbee2/ollama_full/lib/ollama/cuda_v12/libcudart.so.12.8.90
/home/kbarbee2/ollama_full/lib/ollama/cuda_v12/libggml-cuda.so
/home/kbarbee2/ollama_full/lib/ollama/cuda_v13/libcublas.so.13.1.0.3
/home/kbarbee2/ollama_full/lib/ollama/cuda_v13/libcublasLt.so.13.1.0.3
/home/kbarbee2/ollama_full/lib/ollama/cuda_v13/libcudart.so.13.0.96
/home/kbarbee2/ollama_full/lib/ollama/cuda_v13/libggml-cuda.so
/home/kbarbee2/ollama_full/lib/ollama/libggml-base.so.0.0.0
/home/kbarbee2/ollama_full/lib/ollama/libggml-cpu-alderlake.so
/home/kbarbee2/ollama_full/lib/ollama/libggml-cpu-haswell.so
/home/kbarbee2/ollama_full/lib/ollama/libggml-cpu-icelake.so
/home/kbarbee2/ollama_full/lib/ollama/libggml-cpu-sandybridge.so
/home/kbarbee2/ollama_full/lib/ollama/libggml-cpu-skylakex.so
/home/kbarbee

## Cell 3: Kill old Ollama and start with correct GPU library paths

In [3]:
import subprocess
import time
import os

HOME = os.environ['HOME']
OLLAMA_DIR = f'{HOME}/ollama_full'
OLLAMA_PATH = f'{OLLAMA_DIR}/bin/ollama'

# Make sure binary is executable
subprocess.run(['chmod', '+x', OLLAMA_PATH])

# Kill existing
subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(3)

# Build LD_LIBRARY_PATH to include both the ollama libs and system CUDA
cuda_lib_paths = [
    f'{OLLAMA_DIR}/lib/ollama',          # ollama's own bundled cuda runners
    f'{OLLAMA_DIR}/lib',
    '/usr/local/cuda/lib64',             # system CUDA
    '/usr/local/cuda-12.8/lib64',        # versioned CUDA
    '/usr/lib/x86_64-linux-gnu',
    os.environ.get('LD_LIBRARY_PATH', '')
]
ld_path = ':'.join(p for p in cuda_lib_paths if p)

env = {
    **os.environ,
    'OLLAMA_HOME': f'{HOME}/.ollama',
    'OLLAMA_HOST': '127.0.0.1:11434',
    'OLLAMA_NUM_GPU': '99',
    'LD_LIBRARY_PATH': ld_path,
    'PATH': f"{OLLAMA_DIR}/bin:" + os.environ.get('PATH', ''),
}

proc = subprocess.Popen(
    [OLLAMA_PATH, 'serve'],
    env=env,
    stdout=open(f'{HOME}/ollama.log', 'w'),
    stderr=subprocess.STDOUT
)
time.sleep(8)
print(f'Started PID: {proc.pid}')

# Print log immediately
with open(f'{HOME}/ollama.log') as f:
    print(f.read())

Started PID: 280
time=2026-04-03T19:03:37.379Z level=INFO source=routes.go:1742 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/home/kbarbee2/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file:/

## Cell 4: Verify GPU is being used

In [4]:
import requests

# Check server is up
r = requests.get('http://localhost:11434')
print('Server:', r.text)

# Load a model and check VRAM
print('Loading llava:7b...')
r = requests.post(
    'http://localhost:11434/api/generate',
    json={'model': 'llava:7b', 'prompt': 'hi', 'stream': False},
    timeout=120
)
print('Response:', r.json().get('response', r.json()))

# Check VRAM
r = requests.get('http://localhost:11434/api/ps')
for m in r.json().get('models', []):
    print(f"{m['name']}: size_vram={m['size_vram']} bytes")
    if m['size_vram'] > 0:
        print('  ✓ GPU is being used!')
    else:
        print('  ✗ Still using CPU — check log')

Server: Ollama is running
Loading llava:7b...
Response:  Hello! How can I help you today? Is there something on your mind that you would like to talk about? I'm here to listen and provide any assistance or guidance that I can. 
llava:7b: size_vram=10577569792 bytes
  ✓ GPU is being used!


## Cell 5: If still not working — print full log for diagnosis

In [5]:
import os
with open(f"{os.environ['HOME']}/ollama.log") as f:
    print(f.read())

time=2026-04-03T19:03:37.379Z level=INFO source=routes.go:1742 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/home/kbarbee2/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vsco

## Cell 6: Quick model test once GPU is confirmed working

In [6]:
import requests
import base64, io, os
from PIL import Image

BASE_DIR = '/home/kbarbee2/.cache/huggingface/hub/datasets--ReadingTimeMachine--visual_qa_histograms/snapshots/f7ba35f3751cf2b27c943ccb1d313424897c913b/example_hists'
IMG_DIR = os.path.join(BASE_DIR, 'imgs/imgs')

img_file = os.listdir(IMG_DIR)[0]
image = Image.open(os.path.join(IMG_DIR, img_file)).convert('RGB')

buffer = io.BytesIO()
image.save(buffer, format='PNG')
b64 = base64.b64encode(buffer.getvalue()).decode()

for model in ['llava:7b', 'qwen2.5vl:7b']:
    print(f'--- {model} ---')
    r = requests.post(
        'http://localhost:11434/api/generate',
        json={
            'model': model,
            'prompt': 'How many bars are in this histogram? Reply with just a number.',
            'images': [b64],
            'stream': False
        },
        timeout=120
    )
    print(r.json().get('response', r.json()))
    print()

--- llava:7b ---
 3 

--- qwen2.5vl:7b ---
20

